In [1]:
# !pip install pyspark

In [2]:
# !pip install deltalake

In [4]:
# from pyspark.sql import SparkSession
# import os

# # def create_spark_session():
# #     return (
# #         SparkSession.builder
# #             .appName("Data Ingestion to Bronze with Delta")
# #             # Use this if online:
# #             .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
# #             # Or use this if offline:
# #             # .config("spark.jars", "/path/to/jars/delta-spark_2.12-3.2.0.jar,/path/to/jars/scala-library-2.12.15.jar")
# #             .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
# #             .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
# #             .config("spark.sql.shuffle.partitions", "8")
# #             .config("spark.sql.files.maxPartitionBytes", "134217728")
# #             .getOrCreate()
# #     )

# # spark = create_spark_session()


# spark = SparkSession.builder \
#     .appName("table_exists") \
#     .master("local[*]") \
#     .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
#     .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
#     .config("spark.sql.sources.parallelPartitionDiscovery.parallelism", "4") \
#     .getOrCreate()

In [2]:
!pyspark --version

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/10/06 11:31:47 WARN Utils: Your hostname, codespaces-84f801, resolves to a loopback address: 127.0.0.1; using 10.0.2.144 instead (on interface eth0)
25/10/06 11:31:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 4.0.1
      /_/
                        
Using Scala version 2.13.16, OpenJDK 64-Bit Server VM, 21.0.7
Branch HEAD
Compiled by user runner on 2025-09-02T03:10:51Z
Revision 29434ea766b0fc3c3bf6eaadb43a8f931133649e
Url https://github.com/apache/spark
Type --help for more information.


In [4]:
!spark

/bin/bash: line 1: spark: command not found


In [1]:
import os

# Get current working directory (where the notebook is running)
cwd = os.getcwd()

# Go 2 levels up and into a specific folder
target_dir = os.path.abspath(os.path.join(cwd, '..', '..', 'data', 'bronze'))

print(target_dir)

/workspaces/Spark-Delta-Lake-Practice-with-Medallion-Architecture/data/bronze


In [2]:
from pyspark.sql import SparkSession

Initialize Spark session
spark = (
    SparkSession.builder.appName("Data Ingestion to Bronze with Delta")
            # Use this if online:
            .config("spark.jars.packages", "io.delta:delta-spark_2.13:4.0.0")
            # Or use this if offline:
            # .config("spark.jars", "/path/to/jars/delta-spark_2.12-3.2.0.jar,/path/to/jars/scala-library-2.12.15.jar")
            .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
            .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
            # .config("spark.sql.shuffle.partitions", "8")
            # .config("spark.sql.files.maxPartitionBytes", "134217728")
            .getOrCreate()
)





# Path to your local CSV file
file_path = "../../data/raw/ecoride_customers.csv"   # e.g., "C:/data/customers.csv" or "/home/user/data/customers.csv"

# Read CSV into DataFrame
df = spark.read.option("header", "true").option("inferSchema", "true").csv(file_path)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/10/08 10:29:26 WARN Utils: Your hostname, codespaces-84f801, resolves to a loopback address: 127.0.0.1; using 10.0.1.248 instead (on interface eth0)
25/10/08 10:29:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/usr/local/python/3.12.1/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/codespace/.ivy2.5.2/cache
The jars for the packages stored in: /home/codespace/.ivy2.5.2/jars
io.delta#delta-spark_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f8327598-7eb3-4fb7-ab82-ba59b361f2c2;1.0
	confs: [default]
	found io.delta#delta-spark_2.13;4.0.0 in central
	found io.delta#delta-storage;4.0.0 in central
	found org.antlr#antlr4-runtime;4.13.1 in central
:: resolution report :: resolve 176ms :: artifacts dl 8ms
	:: module

In [3]:
df.show()

+---+----------+----------+--------------------+------------+--------------------+---------------+-------------+-------------+
| id|first_name| last_name|               email|       phone|             address|           city|        state|      country|
+---+----------+----------+--------------------+------------+--------------------+---------------+-------------+-------------+
|  1|     Benny|  Heyfield|bheyfield0@cargoc...|405-334-5985|      36 Dapin Trail|  Oklahoma City|     Oklahoma|United States|
|  2|   Glennis| Lightning|glightning1@rakut...|317-380-6675|  467 Hudson Terrace|   Indianapolis|      Indiana|United States|
|  3|   Justine|     Rowth|jrowth2@netscape.com|763-883-8301|    63226 Chive Hill|    Minneapolis|    Minnesota|United States|
|  4|Bartholemy|    Sancho|  bsancho3@prlog.org|940-792-2878|      36236 3rd Lane|  Wichita Falls|        Texas|United States|
|  5|  Demetris|  Worviell|dworviell4@senate...|480-397-2140|   9 Montana Parkway|           Mesa|      Arizona

In [10]:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)



In [21]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("country") \
    .save(target_dir+'/customers_partitioned')


In [36]:
# df2 = spark.read.format("delta").load(target_dir)

In [14]:
df.createOrReplaceTempView("customers")

In [7]:
# spark.sql("select * from customers").show()

In [43]:
# print(target_dir)

/workspaces/Spark-Delta-Lake-Practice-with-Medallion-Architecture/data/bronze


In [9]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS bronze LOCATION '{target_dir}'")
spark.sql("SHOW DATABASES").show()

+---------+
|namespace|
+---------+
|   bronze|
|  default|
+---------+



In [11]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS bronze.people (
     id INTEGER,
     first_name STRING,
     last_name STRING,
     email STRING,
     phone STRING,
     address STRING,
     city STRING,
     state STRING,
     country STRING
)
USING DELTA
LOCATION '{target_dir}'
""")


DataFrame[]

In [16]:
spark.sql("INSERT INTO bronze.people SELECT * FROM customers")

DataFrame[]

In [17]:
spark.sql("SELECT * FROM bronze.people").show()

25/10/08 09:04:53 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+---+----------+----------+--------------------+------------+--------------------+---------------+-------------+-------------+
| id|first_name| last_name|               email|       phone|             address|           city|        state|      country|
+---+----------+----------+--------------------+------------+--------------------+---------------+-------------+-------------+
|  1|     Benny|  Heyfield|bheyfield0@cargoc...|405-334-5985|      36 Dapin Trail|  Oklahoma City|     Oklahoma|United States|
|  2|   Glennis| Lightning|glightning1@rakut...|317-380-6675|  467 Hudson Terrace|   Indianapolis|      Indiana|United States|
|  3|   Justine|     Rowth|jrowth2@netscape.com|763-883-8301|    63226 Chive Hill|    Minneapolis|    Minnesota|United States|
|  4|Bartholemy|    Sancho|  bsancho3@prlog.org|940-792-2878|      36236 3rd Lane|  Wichita Falls|        Texas|United States|
|  5|  Demetris|  Worviell|dworviell4@senate...|480-397-2140|   9 Montana Parkway|           Mesa|      Arizona

In [18]:
spark.sql("DESCRIBE HISTORY bronze.people").show(truncate=False)

+-------+-----------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp              |userId|userName|operation|operationParameters                   |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                                                                                 |userMetadata|engineInfo                         |
+-------+-----------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------------------------------------------------------------+------------+-----------------------------------+
|2      |2025-10-08

# Minio Integration

In [1]:
import os
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("DeltaMedallionMinIO")
    # Delta configs
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # Required packages
    .config(
        "spark.jars.packages",
        ",".join([
            "io.delta:delta-spark_2.13:4.0.0",
            "org.apache.hadoop:hadoop-aws:3.4.0",
            "com.amazonaws:aws-java-sdk-bundle:1.12.262"
        ])
    )
    # MinIO S3A configs
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000")
    .config("spark.hadoop.fs.s3a.access.key", os.getenv("MINIO_ACCESS_KEY", "minioadmin"))
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv("MINIO_SECRET_KEY", "minioadmin"))
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.S3SingleDriverLogStore")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/10/08 11:13:37 WARN Utils: Your hostname, codespaces-84f801, resolves to a loopback address: 127.0.0.1; using 10.0.1.248 instead (on interface eth0)
25/10/08 11:13:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/usr/local/python/3.12.1/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/codespace/.ivy2.5.2/cache
The jars for the packages stored in: /home/codespace/.ivy2.5.2/jars
io.delta#delta-spark_2.13 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3bc7a9f9-339a-4219-bb9d-a42bfec798e0;1.0
	confs: [default]
	found io.delta#delta-spark_2.13;4.0.0 in central
	found io.delta#delta-storage;4.0.0 in central
	found org.

In [2]:
# Path to your local CSV file
file_path = "../../data/raw/ecoride_customers.csv"   # e.g., "C:/data/customers.csv" or "/home/user/data/customers.csv"

# Read CSV into DataFrame
df = spark.read.option("header", "true").option("inferSchema", "true").csv(file_path)

In [3]:
df.show()

+---+----------+----------+--------------------+------------+--------------------+---------------+-------------+-------------+
| id|first_name| last_name|               email|       phone|             address|           city|        state|      country|
+---+----------+----------+--------------------+------------+--------------------+---------------+-------------+-------------+
|  1|     Benny|  Heyfield|bheyfield0@cargoc...|405-334-5985|      36 Dapin Trail|  Oklahoma City|     Oklahoma|United States|
|  2|   Glennis| Lightning|glightning1@rakut...|317-380-6675|  467 Hudson Terrace|   Indianapolis|      Indiana|United States|
|  3|   Justine|     Rowth|jrowth2@netscape.com|763-883-8301|    63226 Chive Hill|    Minneapolis|    Minnesota|United States|
|  4|Bartholemy|    Sancho|  bsancho3@prlog.org|940-792-2878|      36236 3rd Lane|  Wichita Falls|        Texas|United States|
|  5|  Demetris|  Worviell|dworviell4@senate...|480-397-2140|   9 Montana Parkway|           Mesa|      Arizona

In [4]:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)



In [5]:
bronze_path = "s3a://bronze/customers/"
df.write.format("delta").mode("overwrite").save(bronze_path)

25/10/08 11:14:10 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
25/10/08 11:14:13 WARN S3ABlockOutputStream: Application invoked the Syncable API against stream writing to customers/part-00000-32392168-297d-4447-9ee6-708a41907ac4-c000.snappy.parquet. This is Unsupported
                                                                                

In [8]:
df2 = spark.read.format('delta').load(bronze_path)

In [9]:
df2.show()

25/10/08 11:26:32 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+---+----------+----------+--------------------+------------+--------------------+---------------+-------------+-------------+
| id|first_name| last_name|               email|       phone|             address|           city|        state|      country|
+---+----------+----------+--------------------+------------+--------------------+---------------+-------------+-------------+
|  1|     Benny|  Heyfield|bheyfield0@cargoc...|405-334-5985|      36 Dapin Trail|  Oklahoma City|     Oklahoma|United States|
|  2|   Glennis| Lightning|glightning1@rakut...|317-380-6675|  467 Hudson Terrace|   Indianapolis|      Indiana|United States|
|  3|   Justine|     Rowth|jrowth2@netscape.com|763-883-8301|    63226 Chive Hill|    Minneapolis|    Minnesota|United States|
|  4|Bartholemy|    Sancho|  bsancho3@prlog.org|940-792-2878|      36236 3rd Lane|  Wichita Falls|        Texas|United States|
|  5|  Demetris|  Worviell|dworviell4@senate...|480-397-2140|   9 Montana Parkway|           Mesa|      Arizona

In [10]:
base_path = "s3a://lakehousetst/"
os.path.join(base_path, "bronze")

's3a://lakehousetst/bronze'

In [ ]:
from pyspark.sql import SparkSession
import logging
import os

class DataIngestor:
    def __init__(self, spark: SparkSession, base_path: str = "/opt/spark/data/lakehouse"):
        self.spark = spark
        self.base_path = base_path  # root folder for bronze/silver/gold

    def _get_table_path(self, layer: str, business_entity: str, table_name: str) -> str:
        """Construct path for Delta table."""
        return os.path.join(self.base_path, layer, business_entity, table_name)

    def create_or_replace_delta_table(self, df, layer, business_entity, table_name, partition_by=None):
        """Create or overwrite a Delta table."""
        table_path = self._get_table_path(layer, business_entity, table_name)

        try:
            writer = df.write.format("delta").mode("overwrite")
            if partition_by:
                writer = writer.partitionBy(partition_by)
            writer.save(table_path)

            logging.info(f"Delta table created/replaced at {table_path}")
        except Exception as e:
            logging.error(f"Error creating/replacing Delta table: {e}")
            raise e

    def ingest_file_to_bronze(self, file_path: str, business_entity: str, table_name: str, file_type: str, partition_by=None):
        """Ingest raw file into Bronze Delta Lake."""
        try:
            if file_type == 'csv':
                df = self.spark.read.csv(file_path, header=True, inferSchema=True)
            elif file_type == 'json':
                df = self.spark.read.option("multiLine", "true").json(file_path)
            else:
                raise ValueError(f"Unsupported file type '{file_type}'. Supported types: csv, json")

            # Save as Delta in bronze layer
            self.create_or_replace_delta_table(df, "bronze", business_entity, table_name, partition_by)

            logging.info(
                f"Data ingested successfully from {file_path} "
                f"to Delta table bronze/{business_entity}/{table_name}"
            )

        except Exception as e:
            logging.error(f"Error ingesting file to Delta table: {e}")
            raise e
